In [1]:
import os
import urllib.request
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp
import zipfile

url_ceis_zip = "https://portaldatransparencia.gov.br/download-de-dados/favorecidos-pj/202603"
zip_path = "/tmp/favorecidos_pj.zip"
csv_path = "/tmp/favorecidos_pj.csv"


print("Baixando dados de favorecidos PJ...")
urllib.request.urlretrieve(url_ceis_zip, zip_path)

print("Extraindo CSV do ZIP...")
with zipfile.ZipFile(zip_path, "r") as zf:
    csv_files = [name for name in zf.namelist() if name.lower().endswith("cnpj.csv")]
    if not csv_files:
        raise ValueError("Nenhum arquivo com sufixo encontrado no ZIP.")
    extracted_path = zf.extract(csv_files[0], "/tmp")
    os.replace(extracted_path, csv_path)

print(f"CSV extraído em: {csv_path}")

os.environ["AWS_REGION"] = "us-east-1"
os.environ["AWS_ACCESS_KEY_ID"] = "admin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "password"

iceberg_version = "1.4.3"
aws_version = "3.3.4"
packages = [
    f"org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:{iceberg_version}",
    f"org.apache.iceberg:iceberg-aws-bundle:{iceberg_version}",
    f"org.apache.hadoop:hadoop-aws:{aws_version}",
    "com.amazonaws:aws-java-sdk-bundle:1.12.262"
]

spark = SparkSession.builder \
    .appName("Ingestao-CGU-Favorecidos-PJ-Bronze") \
    .config("spark.jars.packages", ",".join(packages)) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.iceberg.type", "hive") \
    .config("spark.sql.catalog.iceberg.uri", "thrift://hive-metastore:9083") \
    .config("spark.sql.catalog.iceberg.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.iceberg.warehouse", "s3a://warehouse/") \
    .config("spark.sql.catalog.iceberg.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.iceberg.client.region", "us-east-1") \
    .config("spark.sql.catalog.iceberg.s3.path-style-access", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
spark.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.bronze")

print("Lendo CSV bruto...")
df_raw = spark.read.csv(csv_path, header=True, sep=";", encoding="ISO-8859-1", inferSchema=True)

df_bronze = df_raw.withColumn("data_ingestao", current_timestamp())

print("Gravando na camada Bronze...")
df_bronze.writeTo("iceberg.bronze.favorecidos_pj") \
    .tableProperty("format-version", "2") \
    .using("iceberg") \
    .createOrReplace()

print("Ingestão Bronze finalizada.")

Baixando dados de favorecidos PJ...
Extraindo CSV do ZIP...
CSV extraído em: /tmp/favorecidos_pj.csv
:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/vscode/.ivy2/cache
The jars for the packages stored in: /home/vscode/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-330e0dd9-3bea-403f-aefc-df37c1e51ee7;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.4.3 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.4.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 180ms :: artifacts dl 7ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default

Lendo CSV bruto...


Gravando na camada Bronze...


Ingestão Bronze finalizada.


In [ ]:
df_show = spark.sql("SELECT * FROM iceberg.bronze.favorecidos_pj limit 10");
df_show.show(truncate=False)